# PetitGPT on Kaggle Tesla T4 with `kaggle-vllm`

## Native inference validation and direct vLLM compatibility boundary

This notebook documents a reproducible compatibility experiment for
[Yang Qi's PetitGPT](https://huggingface.co/yqi0/petitgpt) using
[`kaggle-vllm`](https://github.com/kaggle-vllm/kaggle-vllm) on a Kaggle
**2 × NVIDIA Tesla T4 (SM75)** session.

### What this notebook demonstrates

The executed experiment on **2026-09-17** established:

| Check | Result |
|---|---|
| Kaggle host: Python 3.12.13 / PyTorch 2.10.0+cu128 / CUDA 12.8 | PASS |
| Visible GPUs: 2 × Tesla T4, compute capability 7.5 | PASS |
| `kaggle-vllm==0.2.0` bootstrap and native CUDA extensions | PASS |
| PetitGPT pinned release download + SHA256 verification | PASS |
| PetitGPT architecture / 124,635,456 FP32 parameter inventory | PASS |
| PetitGPT native PyTorch inference (`fp32_math`) on GPU 0 | PASS |
| Transformers `AutoConfig` recognition | FAIL — no recognized `model_type` |
| Direct vLLM TP=1 construction | FAIL — unrecognized model/config |
| vLLM TP=2 | NOT ATTEMPTED — 9 query heads are not divisible by 2 |
| vLLM-native `sharded_state` | NOT CREATED — vLLM construction did not succeed |

**Important:** the successful generation in this notebook is PetitGPT's **native
PyTorch inference path on one T4**. It is not presented as vLLM inference and it
does not claim dual-GPU tensor-parallel execution.

The direct vLLM experiment is intentionally retained because it identifies the
next integration requirement: a proper Transformers/vLLM model integration for
PetitGPT rather than a CUDA-runtime fix.


## 1. Experiment provenance

PetitGPT is pinned to the author's `research-v1` release revision:

`7bf3df96e6880b242b2907d1e68093435feacd75`

The released model card describes a 124.6M-parameter model with a custom
32,000-token vocabulary, 30 transformer layers, hidden width 576, GQA with
9 query heads / 3 KV heads, head dimension 64, RoPE, RMSNorm, SwiGLU, tied
embeddings, 2,048-token context, and FP32 stored weights.

The author currently documents the release as a **native PyTorch** checkpoint.
Transformers `AutoModel`, vLLM, GGUF, ONNX, llama.cpp, and CPU inference are
not claimed as implemented/validated release paths.

This notebook therefore treats the native implementation as the reference path
and tests vLLM separately as a compatibility probe.


## 2. Bootstrap the validated `kaggle-vllm` CUDA runtime

This is the only installation/bootstrap path in the notebook.

Requirements:

- Kaggle accelerator: **GPU T4 ×2**
- Kaggle Internet enabled
- Kaggle Secret named `HF_TOKEN`
- Fresh Python kernel before importing `vllm`

`kaggle-vllm` installs as a lightweight SDK. Its strict bootstrap then stages
the immutable CUDA-enabled vLLM wheel and dependency overlay without replacing
Kaggle's system PyTorch.


In [ ]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

from kaggle_secrets import UserSecretsClient

SDK_VERSION = "0.2.0"

WORK = Path("/kaggle/working")
RUNTIME = WORK / "kaggle-vllm-runtime"
CACHE = WORK / "kaggle-vllm-cache"
HF_HOME = WORK / "hf-cache"

STAGED = RUNTIME / "staged"
OVERLAY = RUNTIME / "overlay"
MANIFEST = RUNTIME / "runtime.json"

token = UserSecretsClient().get_secret("HF_TOKEN")
assert token, "Configure HF_TOKEN in Kaggle Secrets."

os.environ["HF_TOKEN"] = token
os.environ["HUGGING_FACE_HUB_TOKEN"] = token
os.environ["HF_HOME"] = str(HF_HOME)

CACHE.mkdir(parents=True, exist_ok=True)
HF_HOME.mkdir(parents=True, exist_ok=True)

subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "--no-cache-dir",
        f"kaggle-vllm[hub]=={SDK_VERSION}",
    ],
    check=True,
)

import kaggle_vllm
assert kaggle_vllm.__version__ == SDK_VERSION

if RUNTIME.exists():
    shutil.rmtree(RUNTIME)
RUNTIME.mkdir(parents=True)

boot = [
    "kaggle-vllm", "bootstrap", "--strict",
    "--staged", str(STAGED),
    "--overlay", str(OVERLAY),
    "--cache", str(CACHE),
    "--manifest", str(MANIFEST),
]

subprocess.run(boot + ["--dry-run"], check=True)
subprocess.run(boot, check=True)

assert MANIFEST.is_file()

from kaggle_vllm import activate_runtime
assert activate_runtime(MANIFEST)

runtime = json.loads(MANIFEST.read_text(encoding="utf-8"))
RUN_ENV = dict(os.environ)
RUN_ENV.update(runtime["runtime_environment"])

import vllm
import vllm._C
import vllm._moe_C

print("kaggle-vllm:", kaggle_vllm.__version__)
print("vLLM:", vllm.__version__)
print("vLLM path:", Path(vllm.__file__).resolve())
print("vllm._C:", Path(vllm._C.__file__).resolve())
print("vllm._moe_C:", Path(vllm._moe_C.__file__).resolve())

subprocess.run(
    ["kaggle-vllm", "doctor", "--strict"],
    env=RUN_ENV,
    check=True,
)


### Recorded runtime evidence from the executed run

The stopped Kaggle session produced this validated runtime identity:

- profile: `kaggle-t4x2-cu128`
- Python: `3.12.13`
- PyTorch: `2.10.0+cu128`
- PyTorch CUDA: `12.8`
- GPUs: `2 × Tesla T4`
- compute capability: `SM75`
- NCCL: `2.27.5`
- native wheel: `vllm-0.18.2.dev0+ga26e8dc7f.d20260822.cu128-cp312-cp312-linux_x86_64.whl`
- native wheel SHA256: `5a9bd710b8a19fdd23abb3442baad892da977466f996334decd533a225f5fd0c`
- immutable binary revision: `f6b4f10de54924ed6fe9e28cceab84eca7276ab6`

The runtime manifest reported all strict host checks as PASS.


## 3. Download and verify PetitGPT

The model is downloaded at the exact pinned revision used in the experiment.
The author's `SHA256SUMS` file is verified before any model execution.


In [ ]:
from huggingface_hub import snapshot_download

PETITGPT_REPO = "yqi0/petitgpt"
PETITGPT_REVISION = "7bf3df96e6880b242b2907d1e68093435feacd75"

PETITGPT_WORK = Path("/kaggle/working/petitgpt")
MODEL_DIR = PETITGPT_WORK / "petitgpt-research-v1"
NATIVE_LOG = PETITGPT_WORK / "petitgpt-native-fp32.log"
VLLM_LOG = PETITGPT_WORK / "petitgpt-vllm-tp1.log"

PETITGPT_WORK.mkdir(parents=True, exist_ok=True)

if MODEL_DIR.exists():
    shutil.rmtree(MODEL_DIR)

snapshot_path = snapshot_download(
    repo_id=PETITGPT_REPO,
    revision=PETITGPT_REVISION,
    local_dir=str(MODEL_DIR),
    token=token,
)

MODEL_DIR = Path(snapshot_path).resolve()

check = subprocess.run(
    ["sha256sum", "-c", "SHA256SUMS"],
    cwd=MODEL_DIR,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

print(check.stdout)
assert check.returncode == 0
print("PetitGPT release checksum verification: PASS")


## 4. Validate the checkpoint architecture and tensor-parallel constraint

The released config is checked directly rather than inferred from a model name.

The key tensor-parallel observation is straightforward:

- TP=1: `9 % 1 == 0` → valid head partition
- TP=2: `9 % 2 != 0` → invalid head partition

For that reason, this notebook **does not attempt TP=2 for PetitGPT** even
though the Kaggle host exposes two T4 GPUs. The second GPU validates the target
`kaggle-vllm` environment, but the PetitGPT checkpoint itself is tested at TP=1.


In [ ]:
PETIT_CONFIG = json.loads(
    (MODEL_DIR / "config.json").read_text(encoding="utf-8")
)

expected = {
    "vocab_size": 32000,
    "n_layers": 30,
    "d_model": 576,
    "n_heads": 9,
    "n_kv_heads": 3,
    "d_ff": 1536,
    "max_seq_len": 2048,
}

for key, value in expected.items():
    assert PETIT_CONFIG.get(key) == value, (key, PETIT_CONFIG.get(key), value)

HEAD_DIM = PETIT_CONFIG["d_model"] // PETIT_CONFIG["n_heads"]
assert HEAD_DIM == 64

print(json.dumps(PETIT_CONFIG, indent=2))
print("head_dim:", HEAD_DIM)
print("TP=1 valid:", PETIT_CONFIG["n_heads"] % 1 == 0)
print("TP=2 valid:", PETIT_CONFIG["n_heads"] % 2 == 0)

assert PETIT_CONFIG["n_heads"] % 1 == 0
assert PETIT_CONFIG["n_heads"] % 2 != 0


## 5. Verify the exported FP32 tensor inventory

This reads the `safetensors` file directly on CPU and confirms that the released
checkpoint contains **124,635,456 tensor elements**, matching the published
unique parameter count. No weights are transformed in this step.


In [ ]:
from safetensors import safe_open

WEIGHT_FILE = MODEL_DIR / "model.safetensors"
tensor_inventory = []

with safe_open(WEIGHT_FILE, framework="pt", device="cpu") as f:
    for name in f.keys():
        tensor = f.get_tensor(name)
        tensor_inventory.append({
            "name": name,
            "shape": list(tensor.shape),
            "dtype": str(tensor.dtype),
            "numel": tensor.numel(),
        })

total_elements = sum(item["numel"] for item in tensor_inventory)

print("Named tensor entries:", len(tensor_inventory))
print("Tensor elements:", f"{total_elements:,}")
print("Stored dtype(s):", sorted({item["dtype"] for item in tensor_inventory}))

assert total_elements == 124_635_456
assert {item["dtype"] for item in tensor_inventory} == {"torch.float32"}

print("PetitGPT tensor inventory: PASS")


## 6. Reference execution: PetitGPT native PyTorch inference on one T4

This is the model author's supported inference path and is intentionally kept
separate from the vLLM experiment.

`CUDA_VISIBLE_DEVICES=0` restricts the reference run to **one Tesla T4**.
The `fp32_math` profile is used as a conservative SM75-compatible baseline.

The prompt is diagnostic rather than an accuracy benchmark. The generated text
can be factually wrong; this step tests model execution and export integrity,
not answer quality.


In [1]:
native_env = RUN_ENV.copy()
native_env["CUDA_VISIBLE_DEVICES"] = "0"

cmd = [
    sys.executable,
    str(MODEL_DIR / "inference.py"),
    "--model-directory", str(MODEL_DIR),
    "--prompt", "Explain tensor parallelism in one short sentence.",
    "--profile", "fp32_math",
    "--max-new-tokens", "48",
]

native_proc = subprocess.run(
    cmd,
    cwd=MODEL_DIR,
    env=native_env,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

NATIVE_LOG.write_text(native_proc.stdout, encoding="utf-8")
print(native_proc.stdout)
print("Return code:", native_proc.returncode)

assert native_proc.returncode == 0
print("PETITGPT NATIVE PYTORCH INFERENCE: PASS")


{"prompt_token_ids": [2, 5, 13245, 422, 8661, 7701, 916, 293, 616, 1920, 6002, 20, 6], "generated_token_ids": [11060, 7701, 916, 322, 272, 2728, 292, 265, 2164, 2119, 300, 340, 8664, 293, 7701, 20, 672, 519, 265, 6060, 2363, 293, 2754, 14000, 299, 322, 825, 293, 868, 3946, 18, 1372, 2119, 5217, 18, 3261, 14475, 18, 299, 655, 42, 12572, 20, 3], "output_text_for_scoring": "Tensor parallelism is the ability of a single image to be processed in parallel. It's a fundamental concept in computer graphics and is used in many applications, including image processing, video editing, and 3D modeling.", "output_text_raw_including_terminal_eos": "Tensor parallelism is the ability of a single image to be processed in parallel. It's a fundamental concept in computer graphics and is used in many applications, including image processing, video editing, and 3D modeling.[EOS]", "stop_reason": "eos", "generated_tokens_including_eos": 44, "empty_output": false, "repeated_4gram_fraction": 0.0, "unexpected_c

### Recorded native result

The executed run returned code `0`, generated `44` tokens including EOS, and
terminated with `stop_reason="eos"`.

The recorded decoded text was:

> Tensor parallelism is the ability of a single image to be processed in parallel. It's a fundamental concept in computer graphics and is used in many applications, including image processing, video editing, and 3D modeling.

This output is preserved as execution evidence only; it is **not** asserted to
be a correct explanation of tensor parallelism.


## 7. Transformers compatibility probe

Before asking vLLM to construct the model, test whether the released directory
is recognized by Hugging Face `AutoConfig`.

This is expected to fail for the current release because its `config.json`
describes the native PetitGPT implementation rather than a registered
Transformers model contract.


In [ ]:
from transformers import AutoConfig

probe = {
    "success": False,
    "config_class": None,
    "model_type": None,
    "architectures": None,
    "error_type": None,
    "error": None,
}

try:
    cfg = AutoConfig.from_pretrained(
        str(MODEL_DIR),
        trust_remote_code=False,
    )
    probe["success"] = True
    probe["config_class"] = type(cfg).__name__
    probe["model_type"] = getattr(cfg, "model_type", None)
    probe["architectures"] = getattr(cfg, "architectures", None)
except Exception as exc:
    probe["error_type"] = type(exc).__name__
    probe["error"] = str(exc)

print(json.dumps(probe, indent=2))


## 8. Direct vLLM TP=1 compatibility probe

This child-process test uses the already validated `kaggle-vllm` runtime and
tries to construct the exact PetitGPT checkpoint with vLLM at TP=1.

The process is isolated so a model-loader error cannot corrupt the notebook
kernel. A failure here is interpreted according to the actual stack trace.

No TP=2 attempt is made because PetitGPT has 9 query heads.


In [2]:
child_code = r"""
import json
import sys
from vllm import LLM, SamplingParams

model_path = sys.argv[1]

print("Starting PetitGPT vLLM TP=1 load...", flush=True)

llm = LLM(
    model=model_path,
    tensor_parallel_size=1,
    dtype="float16",
    max_model_len=2048,
    gpu_memory_utilization=0.40,
    enforce_eager=True,
    disable_custom_all_reduce=True,
    trust_remote_code=False,
)

params = SamplingParams(temperature=0.0, max_tokens=48)
outputs = llm.generate(
    ["Explain tensor parallelism in one short sentence."],
    params,
)

print("PETITGPT_VLLM_RESULT=" + json.dumps({
    "success": True,
    "text": outputs[0].outputs[0].text,
}))
"""

test_env = dict(RUN_ENV)
test_env["CUDA_VISIBLE_DEVICES"] = "0"
test_env["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

proc = subprocess.run(
    [sys.executable, "-c", child_code, str(MODEL_DIR)],
    env=test_env,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

VLLM_LOG.write_text(proc.stdout, encoding="utf-8")
print(proc.stdout)
print("Return code:", proc.returncode)

VLLM_PASSED = proc.returncode == 0
print("PETITGPT vLLM TP=1:", "PASS" if VLLM_PASSED else "NOT LOADABLE BY CURRENT RELEASE")


Starting PetitGPT vLLM TP=1 load...
INFO 09-17 10:38:18 [utils.py:233] non-default args: {'dtype': 'float16', 'max_model_len': 2048, 'gpu_memory_utilization': 0.4, 'disable_log_stats': True, 'enforce_eager': True, 'disable_custom_all_reduce': True, 'model': '/kaggle/working/petitgpt/petitgpt-research-v1'}
Traceback (most recent call last):
  File "<string>", line 11, in <module>
  File "/kaggle/working/kaggle-vllm-runtime/staged/vllm/entrypoints/llm.py", line 382, in __init__
    self.llm_engine = LLMEngine.from_engine_args(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/kaggle-vllm-runtime/staged/vllm/v1/engine/llm_engine.py", line 169, in from_engine_args
    vllm_config = engine_args.create_engine_config(usage_context)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/kaggle-vllm-runtime/staged/vllm/engine/arg_utils.py", line 1508, in create_engine_config
    model_config = self.create_model_config()
                

### Interpretation of the vLLM failure

The recorded stack trace stops during vLLM `ModelConfig` construction with:

`Unrecognized model ... Should have a model_type key in its config.json`

That is a **model/config registration boundary**. It happens before model
weights are executed on CUDA.

Therefore this run does **not** provide evidence of a failure in:

- the SM75 native CUDA wheel,
- Triton attention,
- NCCL,
- T4 memory capacity,
- FP16 execution,
- or tensor-parallel communication.

It shows that the current PetitGPT release needs a proper model integration
layer before vLLM can construct it.

The notebook deliberately does not edit `config.json` to pretend PetitGPT is a
Llama model. Similar architectural ingredients are not sufficient evidence of
weight/layout/semantic compatibility.


## 9. Why no vLLM `sharded_state` is created

A vLLM-native `sharded_state` is an output of a successfully constructed vLLM
model. Since the exact released PetitGPT checkpoint does not pass model
construction, producing a sharded state would be premature.

The next engineering step is one of:

1. a proper Transformers-compatible `PetitGPTConfig` /
   `PetitGPTForCausalLM` implementation plus vLLM support, or
2. an out-of-tree/custom vLLM model registration and exact weight loader for
   PetitGPT.

Either route must preserve the released architecture and checkpoint semantics.


## 10. Save the machine-readable compatibility report

The report makes the distinction between environment validation, native
inference, model-loader compatibility, TP policy, and sharding explicit.


In [3]:
REPORT_PATH = PETITGPT_WORK / "petitgpt-kaggle-vllm-compatibility-report.json"

report = {
    "petitgpt_repo": PETITGPT_REPO,
    "petitgpt_revision": PETITGPT_REVISION,
    "runtime": {
        "kaggle_vllm_sdk": SDK_VERSION,
        "vllm_version": getattr(vllm, "__version__", "unknown"),
        "cuda": "12.8",
        "gpu_environment": "2 x NVIDIA Tesla T4",
        "compute_capability": "SM75",
    },
    "petitgpt": {
        "parameters": 124_635_456,
        "layers": 30,
        "hidden_size": 576,
        "query_heads": 9,
        "kv_heads": 3,
        "head_dim": 64,
        "max_seq_len": 2048,
    },
    "results": {
        "native_pytorch_fp32_gpu0": "PASS",
        "transformers_autoconfig": "FAIL_UNRECOGNIZED_MODEL",
        "vllm_tp1_load": "FAIL_UNRECOGNIZED_MODEL",
        "vllm_tp2_attempted": False,
        "vllm_tp2_reason": (
            "PetitGPT has 9 query attention heads; "
            "9 is not divisible by tensor_parallel_size=2."
        ),
        "sharded_state_created": False,
        "sharded_state_reason": (
            "vLLM model construction must succeed before a "
            "vLLM-native sharded_state can be created."
        ),
    },
    "conclusion": (
        "PetitGPT runs successfully with its native PyTorch inference "
        "implementation on Kaggle Tesla T4. Direct loading through the "
        "current kaggle-vllm/vLLM runtime fails at model architecture "
        "recognition because the released checkpoint does not expose a "
        "Transformers/vLLM-recognized model_type/model integration."
    ),
}

REPORT_PATH.write_text(json.dumps(report, indent=2), encoding="utf-8")
print(json.dumps(report, indent=2))
print("Saved:", REPORT_PATH)


{
  "petitgpt_repo": "yqi0/petitgpt",
  "petitgpt_revision": "7bf3df96e6880b242b2907d1e68093435feacd75",
  "runtime": {
    "kaggle_vllm_sdk": "0.2.0",
    "vllm_version": "0.18.2.dev0+ga26e8dc7f.d20260822",
    "cuda": "12.8",
    "gpu_environment": "2 x NVIDIA Tesla T4",
    "compute_capability": "SM75"
  },
  "petitgpt": {
    "parameters": 124635456,
    "layers": 30,
    "hidden_size": 576,
    "query_heads": 9,
    "kv_heads": 3,
    "head_dim": 64,
    "max_seq_len": 2048
  },
  "results": {
    "native_pytorch_fp32_gpu0": "PASS",
    "transformers_autoconfig": "FAIL_UNRECOGNIZED_MODEL",
    "vllm_tp1_load": "FAIL_UNRECOGNIZED_MODEL",
    "vllm_tp2_attempted": false,
    "vllm_tp2_reason": "PetitGPT has 9 query attention heads; 9 is not divisible by tensor_parallel_size=2.",
    "sharded_state_created": false,
    "sharded_state_reason": "vLLM model construction must succeed before a vLLM-native sharded_state can be created."
  },
  "conclusion": "PetitGPT runs successfully with

## 11. Reproduction result and collaboration opportunity

This experiment establishes a useful boundary:

- **Portable today:** the released PetitGPT native PyTorch implementation runs
  on a Kaggle Tesla T4 under the validated CUDA 12.8 environment.
- **Not portable by direct loading today:** the exact released checkpoint is
  not a registered Transformers/vLLM model.
- **Not a CUDA blocker:** the direct vLLM attempt stops at model recognition,
  before CUDA model execution.
- **TP=2 is not a valid direct target for this architecture:** 9 query heads
  cannot be evenly partitioned across two ranks.

A natural follow-up experiment would be a faithful PetitGPT model adapter for
vLLM, followed by TP=1 performance measurements on T4 and, only after successful
vLLM construction, a persistent `sharded_state` export/reload test.

### Attribution

PetitGPT is by **Yang Qi**. Please consult the upstream model card, source
repository, licenses, provenance, and citation record before redistributing
upstream files or results.

- Model: https://huggingface.co/yqi0/petitgpt
- Source: https://github.com/yangqi0/petitgpt
- `kaggle-vllm`: https://github.com/kaggle-vllm/kaggle-vllm
